# Jelastic Billing Export

This notebook connects to your **SaveInCloud / Jelastic** account, exports billing data using the native CSV export API (same format as the portal), and optionally uploads via SFTP.

**CSV columns (same as SaveInCloud portal):**
Date, Env, Node Id, Item, Type, Note, Reserved Cloudlets, Free Reserved Cloudlets, Dynamic Cloudlets, Free Dynamic Cloudlets, Paid Storage (MB), Free Storage (MB), Paid Traffic (MB), Free Traffic (MB), Billable resource, Free resource, Cost (BRL)

**How to use:**
1. Fill in your credentials in Section 1
2. Run all cells (Runtime > Run all)
3. Review the results and download the CSV

---

## 0. Install Dependencies

In [ ]:
!pip install requests paramiko -q

## 1. Configuration

Edit the values below with your credentials.

In [ ]:
# =====================================================
# JELASTIC API SETTINGS
# =====================================================
API_URL = "https://app.paas.saveincloud.net.br/1.0/"
API_TOKEN = "YOUR_TOKEN_HERE"  # <-- PASTE YOUR TOKEN HERE

# =====================================================
# BILLING SETTINGS
# =====================================================
LOOKBACK_DAYS = 7

# Or use fixed dates (leave empty to use LOOKBACK_DAYS)
FIXED_START = ""  # e.g. "2026-05-01 00:00:00"
FIXED_END = ""    # e.g. "2026-05-12 23:59:59"

# Granularity: "HOUR", "DAY", "MONTH"
# Note: HOUR max 1 day range, DAY max 7 days (auto-chunked)
PERIOD = "DAY"

# Group nodes in same environment? True or False
GROUP_NODES = False

# Filter by environment name (empty = all environments)
ENV_NAME = ""  # e.g. "torkes"

# =====================================================
# SFTP UPLOAD (OPTIONAL)
# =====================================================
SFTP_ENABLED = False  # Set to True to enable upload
SFTP_HOST = "YOUR_SFTP_HOST"
SFTP_PORT = 22
SFTP_USER = "YOUR_SFTP_USER"
SFTP_PASS = "YOUR_SFTP_PASS"
SFTP_REMOTE_DIR = "/home/arthur/"

print("Configuration loaded.")

## 2. Setup

In [ ]:
import requests
import csv
import io
from datetime import datetime, timedelta

PERIOD_MAX_DAYS = {"HOUR": 1, "DAY": 7, "MONTH": 365, "YEAR": 3650}

# Compute date range
if FIXED_START and FIXED_END:
    start_dt = datetime.strptime(FIXED_START, "%Y-%m-%d %H:%M:%S")
    end_dt = datetime.strptime(FIXED_END, "%Y-%m-%d %H:%M:%S")
else:
    now = datetime.now()
    end_dt = now.replace(hour=23, minute=59, second=59, microsecond=0)
    start_dt = (now - timedelta(days=LOOKBACK_DAYS)).replace(
        hour=0, minute=0, second=0, microsecond=0
    )

print(f"Date range: {start_dt} to {end_dt}")

session = requests.Session()

def api_call(endpoint, params=None):
    url = f"{API_URL.rstrip('/')}/{endpoint}"
    if params is None:
        params = {}
    params["session"] = API_TOKEN
    resp = session.get(url, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if data.get("result") != 0:
        print(f"  API error: {data.get('error', 'Unknown')}")
        return None
    return data

def chunk_date_range(start, end, period):
    max_days = PERIOD_MAX_DAYS.get(period, 7)
    chunks = []
    current = start
    while current < end:
        chunk_end = min(current + timedelta(days=max_days), end)
        chunks.append((current, chunk_end))
        current = chunk_end + timedelta(seconds=1)
    return chunks

print("Setup complete.")

## 3. List Environments

In [ ]:
data = api_call("environment/control/rest/getenvs")
environments = []

if data:
    for info in data.get("infos", []):
        env = info.get("env", {})
        env_info = {
            "envName": env.get("envName", ""),
            "domain": env.get("domain", ""),
            "status": env.get("status", 0),
        }
        environments.append(env_info)
        status_map = {1: "Running", 2: "Stopped", 3: "Sleeping"}
        print(f"  {env_info['envName']}")
        print(f"    Domain: {env_info['domain']}")
        print(f"    Status: {status_map.get(env_info['status'], str(env_info['status']))}")
        print()

print(f"Total: {len(environments)} environment(s)")

## 4. Export Billing CSV

Uses the native `ExportAccountBillingHistoryByPeriod` API endpoint - the same one the SaveInCloud portal uses. Produces identical CSV columns.

In [ ]:
chunks = chunk_date_range(start_dt, end_dt, PERIOD)
all_lines = []
header_line = None

print(f"Fetching billing data ({len(chunks)} chunk(s))...")

for i, (chunk_start, chunk_end) in enumerate(chunks):
    params = {
        "startTime": chunk_start.strftime("%Y-%m-%d %H:%M:%S"),
        "endTime": chunk_end.strftime("%Y-%m-%d %H:%M:%S"),
        "period": PERIOD,
        "groupNodes": str(GROUP_NODES).lower(),
    }
    if ENV_NAME:
        params["envName"] = ENV_NAME

    result = api_call(
        "billing/account/rest/exportaccountbillinghistorybyperiod",
        params
    )

    if result and result.get("link"):
        resp = session.get(result["link"], timeout=120)
        resp.raise_for_status()
        csv_text = resp.text.strip()

        if csv_text:
            lines = csv_text.split("\n")
            if header_line is None and lines:
                header_line = lines[0]
            data_rows = [l for l in lines[1:] if l.strip()]
            all_lines.extend(data_rows)
            print(f"  Chunk {i+1}/{len(chunks)}: {len(data_rows)} rows")
        else:
            print(f"  Chunk {i+1}/{len(chunks)}: empty")

total_rows = len(all_lines)
print(f"\nTotal billing rows: {total_rows}")

if header_line:
    print(f"\nCSV columns:\n{header_line}")

if total_rows == 0:
    print("\nNo billing data found. This is normal for trial/collaborator accounts.")
    print("Use your production token to see real billing data.")

## 5. Display Billing Data

In [ ]:
if header_line and all_lines:
    try:
        import pandas as pd
        csv_full = header_line + "\n" + "\n".join(all_lines)
        df = pd.read_csv(io.StringIO(csv_full))
        # Clean column names (remove leading/trailing spaces)
        df.columns = [c.strip() for c in df.columns]
        display(df)
        if "Cost (BRL)" in df.columns:
            print(f"\nTotal Cost: R$ {df['Cost (BRL)'].sum():.2f}")
    except ImportError:
        print(header_line)
        for line in all_lines[:50]:
            print(line)
        if len(all_lines) > 50:
            print(f"... and {len(all_lines) - 50} more rows")
else:
    print("No data to display.")

## 6. Save and Download CSV

In [ ]:
csv_filename = f"billing_{datetime.now().strftime('%Y-%m-%d')}.csv"

if header_line:
    csv_content = header_line + "\n"
    if all_lines:
        csv_content += "\n".join(all_lines) + "\n"

    with open(csv_filename, "w", encoding="utf-8") as f:
        f.write(csv_content)

    print(f"CSV saved: {csv_filename} ({len(all_lines)} data rows)")

    # Auto-download in Google Colab
    try:
        from google.colab import files
        files.download(csv_filename)
        print("Download started automatically.")
    except ImportError:
        print(f"File saved locally: {csv_filename}")
else:
    print("No CSV to save (no data or API error).")

## 7. Upload via SFTP (Optional)

Only runs if `SFTP_ENABLED = True` in Section 1.

In [ ]:
if SFTP_ENABLED and header_line:
    import paramiko

    print(f"Connecting to SFTP: {SFTP_USER}@{SFTP_HOST}:{SFTP_PORT}")

    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    try:
        ssh.connect(
            hostname=SFTP_HOST, port=SFTP_PORT,
            username=SFTP_USER, password=SFTP_PASS, timeout=30,
        )
        sftp = ssh.open_sftp()

        remote_path = f"{SFTP_REMOTE_DIR.rstrip('/')}/{csv_filename}"
        sftp.put(csv_filename, remote_path)

        attrs = sftp.stat(remote_path)
        print(f"Uploaded: {csv_filename} -> {remote_path} ({attrs.st_size} bytes)")

        sftp.close()
        ssh.close()
        print("SFTP upload complete!")

    except Exception as e:
        print(f"SFTP upload failed: {e}")
        try:
            ssh.close()
        except:
            pass

elif not SFTP_ENABLED:
    print("SFTP upload disabled. Set SFTP_ENABLED = True in Section 1 to enable.")
else:
    print("No data to upload.")

---

## Next Steps

Once you're satisfied with the output:

1. **For automated weekly runs**, install the standalone script on your Linux VM:
   - Clone: `git clone https://github.com/anirudhatalmale6-alt/jelastic-billing-export.git`
   - Edit `config.yaml` with your credentials
   - Schedule: `crontab -e` and add `0 6 * * 1 cd /path/to/script && python3 jelastic_billing_export.py`

2. **To change the date range**, edit `LOOKBACK_DAYS` or set `FIXED_START`/`FIXED_END` in Section 1.

3. **To filter by environment**, set `ENV_NAME = "your_env_name"` in Section 1.

4. **To get hourly data**, set `PERIOD = "HOUR"` (max 1 day range per chunk).